In [1]:
"""
Temporal Feature Extraction (Simplified)
=========================================
从Reddit数据提取时序特征

输入: reddit_wsb_for_network.csv
输出: temporal_features_5min.parquet

核心特征:
1. Volume (数量)
   - post_volume, comment_volume
   
2. Velocity (速度)
   - volume_velocity (增长速度)
   
3. Acceleration (加速度)
   - volume_acceleration
   
4. Burst Detection (突发检测)
   - is_burst (突发标记)
   - burst_intensity (突发强度)
   
5. User Activity (用户活跃度)
   - unique_users
   - new_users_ratio
"""

import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# ============================================================
# 参数设置
# ============================================================

TIME_WINDOW = 5  # 分钟
DATA_FILE = 'reddit_wsb_for_network.csv'

# Burst检测参数
BURST_THRESHOLD = 3.0  # 超过均值3个标准差视为突发

# ============================================================
# 核心函数
# ============================================================

def extract_temporal_features(reddit_df):
    """
    提取时序特征
    """
    
    print("\nExtracting temporal features...")
    
    # 创建时间窗口
    reddit_df['time_window'] = reddit_df['timestamp'].dt.floor(f'{TIME_WINDOW}min')
    
    # ========== 基础统计 ==========
    
    # 按type分组统计
    volume_stats = reddit_df.groupby(['time_window', 'type']).size().unstack(fill_value=0)
    
    # 确保两列都存在
    if 'post' not in volume_stats.columns:
        volume_stats['post'] = 0
    if 'comment' not in volume_stats.columns:
        volume_stats['comment'] = 0
    
    volume_stats = volume_stats.reset_index()
    volume_stats.columns = ['timestamp', 'post_volume', 'comment_volume']
    
    # 总volume
    volume_stats['total_volume'] = volume_stats['post_volume'] + volume_stats['comment_volume']
    
    # ========== 用户统计 ==========
    
    user_stats = reddit_df.groupby('time_window').agg({
        'user_id': 'nunique',  # 唯一用户数
        'score': ['sum', 'mean', 'max']  # Score统计
    }).reset_index()
    
    user_stats.columns = ['timestamp', 'unique_users', 'total_score', 'avg_score', 'max_score']
    
    # 合并
    features = volume_stats.merge(user_stats, on='timestamp', how='outer').fillna(0)
    
    # 排序
    features = features.sort_values('timestamp').reset_index(drop=True)
    
    # ========== Velocity (速度) ==========
    
    # 计算相对于前一个窗口的变化率
    features['post_velocity'] = features['post_volume'].diff()
    features['comment_velocity'] = features['comment_volume'].diff()
    features['total_velocity'] = features['total_volume'].diff()
    
    # 填充第一行
    features['post_velocity'] = features['post_velocity'].fillna(0)
    features['comment_velocity'] = features['comment_velocity'].fillna(0)
    features['total_velocity'] = features['total_velocity'].fillna(0)
    
    # ========== Acceleration (加速度) ==========
    
    # 速度的变化率
    features['post_acceleration'] = features['post_velocity'].diff().fillna(0)
    features['comment_acceleration'] = features['comment_velocity'].diff().fillna(0)
    features['total_acceleration'] = features['total_velocity'].diff().fillna(0)
    
    # ========== Burst Detection (突发检测) ==========
    
    # 计算滚动统计（60分钟窗口 = 12个5分钟窗口）
    window_size = 12
    
    features['volume_mean_60min'] = features['total_volume'].rolling(
        window=window_size, min_periods=1
    ).mean()
    
    features['volume_std_60min'] = features['total_volume'].rolling(
        window=window_size, min_periods=1
    ).std()
    
    # Z-score
    features['volume_zscore'] = (
        (features['total_volume'] - features['volume_mean_60min']) / 
        (features['volume_std_60min'] + 1e-8)
    )
    
    # Burst标记
    features['is_burst'] = (features['volume_zscore'] > BURST_THRESHOLD).astype(int)
    
    # Burst强度
    features['burst_intensity'] = features['volume_zscore'].clip(lower=0)
    
    # ========== Rolling Features (滚动特征) ==========
    
    # 5分钟、15分钟、30分钟窗口
    for mins in [5, 15, 30]:
        n_windows = mins // TIME_WINDOW
        
        features[f'post_volume_{mins}min'] = features['post_volume'].rolling(
            window=n_windows, min_periods=1
        ).sum()
        
        features[f'comment_volume_{mins}min'] = features['comment_volume'].rolling(
            window=n_windows, min_periods=1
        ).sum()
        
        features[f'total_volume_{mins}min'] = features['total_volume'].rolling(
            window=n_windows, min_periods=1
        ).sum()
    
    # ========== 用户增长率 ==========
    
    # 计算新用户（相对于前一窗口）
    # 简化版：用unique users的变化作为proxy
    features['user_growth'] = features['unique_users'].diff().fillna(0).clip(lower=0)
    
    print(f"  ✓ Extracted temporal features for {len(features):,} windows")
    
    return features


# ============================================================
# 主函数
# ============================================================

def main():
    start_time = datetime.now()
    
    print("\n" + "="*70)
    print("TEMPORAL FEATURE EXTRACTION")
    print("="*70)
    
    # ========== Step 1: 加载数据 ==========
    print("\nStep 1: Loading Reddit data...")
    
    try:
        reddit_df = pd.read_csv(DATA_FILE)
        reddit_df['timestamp'] = pd.to_datetime(reddit_df['timestamp'])
        
        print(f"  ✓ Loaded {len(reddit_df):,} items")
        print(f"  Date range: {reddit_df['timestamp'].min()} to {reddit_df['timestamp'].max()}")
        
        # 检查type列
        if 'type' in reddit_df.columns:
            print(f"\n  Type distribution:")
            for t, count in reddit_df['type'].value_counts().items():
                print(f"    {t}: {count:,}")
        else:
            print(f"  ⚠️  No 'type' column found")
            return
        
    except FileNotFoundError:
        print(f"  ✗ {DATA_FILE} not found!")
        return
    
    # ========== Step 2: 提取特征 ==========
    print("\nStep 2: Extracting temporal features...")
    
    temporal_features = extract_temporal_features(reddit_df)
    
    # ========== Step 3: 统计 ==========
    print("\nStep 3: Feature statistics...")
    
    print(f"\n  Volume statistics:")
    print(f"    Mean post volume: {temporal_features['post_volume'].mean():.1f}/5min")
    print(f"    Mean comment volume: {temporal_features['comment_volume'].mean():.1f}/5min")
    print(f"    Peak total volume: {temporal_features['total_volume'].max():.0f}/5min")
    
    print(f"\n  Burst statistics:")
    bursts = temporal_features[temporal_features['is_burst'] == 1]
    print(f"    Burst windows: {len(bursts):,}")
    if len(bursts) > 0:
        print(f"    Mean burst intensity: {bursts['burst_intensity'].mean():.2f}")
        print(f"    Max burst intensity: {bursts['burst_intensity'].max():.2f}")
    
    print(f"\n  User statistics:")
    print(f"    Mean unique users: {temporal_features['unique_users'].mean():.1f}/5min")
    print(f"    Peak unique users: {temporal_features['unique_users'].max():.0f}/5min")
    
    # ========== Step 4: 保存 ==========
    print("\nStep 4: Saving...")
    
    temporal_features.to_parquet('temporal_features_5min.parquet', index=False)
    
    print(f"  ✓ Saved: temporal_features_5min.parquet")
    print(f"  Shape: {temporal_features.shape}")
    
    # ========== 总结 ==========
    end_time = datetime.now()
    duration = (end_time - start_time).total_seconds()
    
    print("\n" + "="*70)
    print("COMPLETE")
    print("="*70)
    
    print(f"\nTime windows: {len(temporal_features):,}")
    print(f"Feature columns: {len(temporal_features.columns)}")
    
    print(f"\nFeatures extracted:")
    print(f"  ✓ Volume features (post, comment, total)")
    print(f"  ✓ Velocity features (growth rate)")
    print(f"  ✓ Acceleration features")
    print(f"  ✓ Burst detection (z-score based)")
    print(f"  ✓ Rolling windows (5min, 15min, 30min)")
    print(f"  ✓ User activity features")
    
    print(f"\nRuntime: {duration:.1f} seconds")
    
    print("\n✓ Feature extraction complete!")
    print("  (Alignment will be done in merge_all_features.py)")


if __name__ == "__main__":
    main()


TEMPORAL FEATURE EXTRACTION

Step 1: Loading Reddit data...
  ✓ Loaded 1,606,093 items
  Date range: 2019-07-01 04:35:02 to 2021-06-29 23:58:28

  Type distribution:
    comment: 1,394,123
    post: 211,970

Step 2: Extracting temporal features...

Extracting temporal features...
  ✓ Extracted temporal features for 77,267 windows

Step 3: Feature statistics...

  Volume statistics:
    Mean post volume: 18.0/5min
    Mean comment volume: 2.7/5min
    Peak total volume: 1261/5min

  Burst statistics:
    Burst windows: 355
    Mean burst intensity: 3.16
    Max burst intensity: 3.18

  User statistics:
    Mean unique users: 18.8/5min
    Peak unique users: 1087/5min

Step 4: Saving...
  ✓ Saved: temporal_features_5min.parquet
  Shape: (77267, 29)

COMPLETE

Time windows: 77,267
Feature columns: 29

Features extracted:
  ✓ Volume features (post, comment, total)
  ✓ Velocity features (growth rate)
  ✓ Acceleration features
  ✓ Burst detection (z-score based)
  ✓ Rolling windows (5min, 1